# 🍓🐧 Benchmark 02: PyTorch + Custom OpenAI Triton Kernels (`02_triton`) — UnB Metodologia Científica (Trabalho 2)

Bem-vindos ao segundo experimento do nosso estudo comparativo no Google Colab Pro! ✨ Neste notebook, avaliamos o paradigma de **Programação em Blocos (Tile-Based Programming) com OpenAI Triton (`@triton.jit`)** (`implementations/02_triton`) mantendo exatamente a mesma arquitetura **GPT de 10.788.929 parâmetros** e o corpus *TinyShakespeare* (`vocab_size = 65`).

### ⚡ Kernels Customizados em Triton (`@triton.jit`)
- **LayerNorm Forward & Backward:** Redução em registradores por linha com média e variância fundidas.
- **Fused Causal FlashAttention Forward & Backward:** Atenção causal com softmax online em SRAM sem materializar a matriz $T \times T$ na HBM.
- **Fused Cross-Entropy Forward & Backward:** Log-Sum-Exp online e gradiente dos logits calculados em uma única passagem.
- **ReLU Forward & Backward:** Operação elemento-a-elemento vetorizada em blocos.

### 🎯 Invariantes Metodológicos
- **Semente Global Determinística (`SEED`):** `998244353` (mesmos pesos iniciais $\theta_0$ e mesmos mini-batches $X_t, Y_t$!)
- **Precisão Numérica:** `FP32` estrito (`input_precision="ieee"` nos `tl.dot` do Triton e `tf32` desabilitado).

## 🛠️ Célula 1: Verificação da GPU & Setup do Repositório GitHub 🍓

Verificamos a GPU disponível com `!nvidia-smi` e preparamos o diretório do repositório experimental.

In [ ]:
!nvidia-smi

import os
from pathlib import Path

REPO_URL = "https://github.com/<SEU_USUARIO>/<SEU_REPO>.git"
REPO_DIR = Path("trabalho_2")

if Path("common/generate_golden.py").exists() and Path("implementations/02_triton/benchmark.py").exists():
    print("🍓 [Chuu Lab] Repositório já detectado no diretório atual:", Path.cwd())
elif (REPO_DIR / "common/generate_golden.py").exists():
    os.chdir(REPO_DIR)
    print("🍓 [Chuu Lab] Entrando no diretório existente do repositório:", Path.cwd())
else:
    print(f"🐧 [Chuu Lab] Clonando repositório de {REPO_URL} ...")
    !git clone {REPO_URL} {REPO_DIR}
    if REPO_DIR.exists():
        os.chdir(REPO_DIR)
    print("🍓 [Chuu Lab] Diretório ativo:", Path.cwd())


## 🌟 Célula 2: Geração da Referência Dourada (`SEED = 998244353`) & Sanity Check 🐧

Geramos os pesos iniciais $\theta_0$ e os mini-batches determinísticos em `/tmp/golden_gpt`, inspecionando `loss_step_0` e `loss_step_1` para validação de paridade matemática.

In [ ]:
!python3 -m common.generate_golden --output-dir /tmp/golden_gpt --num-steps 5000 --eval-iters 50 --seed 998244353

import json
from pathlib import Path

golden_meta = json.loads(Path("/tmp/golden_gpt/golden_meta.json").read_text(encoding="utf-8"))
print("\n🍓 === Resumo do Golden Meta (SEED = 998244353) === 🐧")
print(json.dumps({
    "seed": golden_meta["seed"],
    "num_parameters": golden_meta["num_parameters"],
    "vocab_size": golden_meta["vocab_size"],
    "loss_step_0": golden_meta["loss_step_0"],
    "loss_step_1": golden_meta["loss_step_1"],
}, indent=2))


## 🚀 Célula 3: Execução do Benchmark Completo (`02_triton` — 5000 Steps + Inferência $B=1$ e $B=64$) 🍓

Rodamos o benchmark de 5000 iterações de treino (após Sanity Check e 50 iterações de Warm-up com compilação JIT do Triton) seguido pela inferência autoregressiva em $B=1$ e $B=64$.

In [ ]:
!python3 implementations/02_triton/benchmark.py --golden-dir /tmp/golden_gpt --output-json results_02_triton.json --max-iters 5000 --warmup-iters 50 --eval-interval 500 --eval-iters 50 --gen-tokens 1000


## 📊 Célula 4: Inspeção das Métricas e Download Automático do JSON 🐧

Inspecionamos os resultados consolidados em `results_02_triton.json` e fazemos o download automático para a máquina local.

In [ ]:
import json
from pathlib import Path

result_path = Path("results_02_triton.json")
report = json.loads(result_path.read_text(encoding="utf-8"))

summary_view = {
    "implementation": report.get("implementation"),
    "gpu_name": report.get("gpu_name"),
    "sanity_check": report.get("sanity_check"),
    "train_time_s": report.get("train_time_s"),
    "train_tokens_per_sec": report.get("train_tokens_per_sec"),
    "infer_b1_tokens_per_sec": report.get("infer_b1_tokens_per_sec"),
    "infer_b1_latency_ms_per_token": report.get("infer_b1_latency_ms_per_token"),
    "infer_b64_tokens_per_sec": report.get("infer_b64_tokens_per_sec"),
    "peak_vram_mb": report.get("peak_vram_mb"),
    "final_train_loss": report.get("final_train_loss"),
    "final_val_loss": report.get("final_val_loss"),
    "sloc_core": report.get("sloc_core"),
    "sloc_harness": report.get("sloc_harness"),
    "sloc_total": report.get("sloc_total"),
}

print("🍓 === Resumo Final: 02_triton === 🐧")
print(json.dumps(summary_view, indent=2))

try:
    from google.colab import files
    files.download("results_02_triton.json")
except ImportError:
    print("🐧 Execução fora do Google Colab detectada; download automático ignorado.")
